# EEMD-TGNet: Evaluation and Results Analysis (Fixed)

This notebook includes the necessary model class definitions at the top to allow unpickling of saved models.

In [10]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("Libraries imported successfully!")

Libraries imported successfully!


In [11]:
# Model class definitions required for pickle loading
class TemporalConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.2):
        super(TemporalConvBlock, self).__init__()
        padding = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation, padding=padding)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, dilation=dilation, padding=padding)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = out[:, :, :-self.conv1.padding[0]]
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.dropout1(out)
        out = self.conv2(out)
        out = out[:, :, :-self.conv2.padding[0]]
        out = self.bn2(out)
        if self.residual is not None:
            residual = self.residual(residual)
        if residual.size(2) != out.size(2):
            residual = residual[:, :, :out.size(2)]
        out = self.relu2(out + residual)
        out = self.dropout2(out)
        return out

class TCN(nn.Module):
    def __init__(self, input_size, num_channels, kernel_size=3, dropout=0.2):
        super(TCN, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation = 2 ** i
            in_channels = input_size if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers.append(TemporalConvBlock(in_channels, out_channels, kernel_size, dilation, dropout))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        x = x.transpose(1, 2)
        out = self.network(x)
        out = out.transpose(1, 2)
        return out

class TCNGRU(nn.Module):
    def __init__(self, input_size, tcn_channels, gru_hidden_size, output_size, tcn_kernel_size=3, tcn_dropout=0.2, gru_dropout=0.2, num_gru_layers=1):
        super(TCNGRU, self).__init__()
        self.input_size = input_size
        self.window_size = None
        self.tcn = TCN(1, tcn_channels, tcn_kernel_size, tcn_dropout)
        tcn_output_size = tcn_channels[-1]
        self.feature_projection = nn.Linear(tcn_output_size + 7, gru_hidden_size)
        self.gru = nn.GRU(gru_hidden_size, gru_hidden_size, num_layers=num_gru_layers, dropout=gru_dropout if num_gru_layers > 1 else 0, batch_first=True)
        self.output_layer = nn.Linear(gru_hidden_size, output_size)
        self.dropout = nn.Dropout(gru_dropout)
    def forward(self, x):
        batch_size = x.size(0)
        if self.window_size is None:
            if x.size(1) == 31:
                self.window_size = 24
            elif x.size(1) == 21:
                self.window_size = 14
            else:
                self.window_size = x.size(1) - 7
        time_series = x[:, :self.window_size]
        additional_features = x[:, self.window_size:]
        time_series = time_series.unsqueeze(-1)
        tcn_out = self.tcn(time_series)
        tcn_features = tcn_out[:, -1, :]
        combined_features = torch.cat([tcn_features, additional_features], dim=1)
        gru_input = self.feature_projection(combined_features)
        gru_input = gru_input.unsqueeze(1)
        gru_out, _ = self.gru(gru_input)
        gru_out = gru_out.squeeze(1)
        gru_out = self.dropout(gru_out)
        output = self.output_layer(gru_out)
        return output

# Add SimpleLSTM class for baseline unpickling
class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1, dropout=0.2):
        super(SimpleLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out

# Add SimpleGRU class for baseline unpickling
class SimpleGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1, dropout=0.2):
        super(SimpleGRU, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out, _ = self.gru(x)
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out

# Add SimpleMLP class for baseline unpickling
class SimpleMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=2, dropout=0.2):
        super(SimpleMLP, self).__init__()
        layers = []
        in_dim = input_size
        for i in range(num_layers - 1):
            layers.append(nn.Linear(in_dim, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = hidden_size
        layers.append(nn.Linear(in_dim, output_size))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

print("Model classes (including SimpleLSTM, SimpleGRU, SimpleMLP) defined for pickle loading!")

Model classes (including SimpleLSTM, SimpleGRU, SimpleMLP) defined for pickle loading!


## 1. Load Training Results

In [12]:
# Load training results
try:
    with open('training_results.pkl', 'rb') as f:
        training_results = pickle.load(f)
    print("Training results loaded successfully!")
    print(f"Available results:")
    for key in training_results.keys():
        print(f"  - {key}")
    gefcom_models = training_results['gefcom_models']
    alibaba_models = training_results['alibaba_models']
    gefcom_baselines = training_results['gefcom_baselines']
    alibaba_baselines = training_results['alibaba_baselines']
except (FileNotFoundError, AttributeError) as e:
    print(f"Training results not found or incompatible: {e}")
    print("Creating dummy results for demonstration...")
    def create_dummy_results(n_imfs, dataset_name):
        models = []
        for i in range(n_imfs):
            base_mse = 0.005 + np.random.normal(0, 0.001)
            base_mae = 0.04 + np.random.normal(0, 0.005)
            base_r2 = 0.94 + np.random.normal(0, 0.02)
            models.append({
                'imf_index': i,
                'test_results': {
                    'mse': max(0.001, base_mse),
                    'mae': max(0.01, base_mae),
                    'r2': min(0.99, max(0.85, base_r2))
                },
                'original_params': np.random.randint(70000, 90000),
                'pruned_params': np.random.randint(40000, 50000),
                'compression_ratio': np.random.uniform(35, 45)
            })
        return models
    def create_dummy_baselines():
        baselines = {}
        models = ['LSTM', 'GRU', 'MLP']
        for model in models:
            mse = 0.008 + np.random.normal(0, 0.002)
            mae = 0.06 + np.random.normal(0, 0.01)
            r2 = 0.88 + np.random.normal(0, 0.03)
            baselines[model] = {
                'test_results': {
                    'mse': max(0.003, mse),
                    'mae': max(0.02, mae),
                    'r2': min(0.95, max(0.80, r2))
                }
            }
        return baselines
    gefcom_models = create_dummy_results(3, "GEFCom")
    alibaba_models = create_dummy_results(2, "Alibaba")
    gefcom_baselines = create_dummy_baselines()
    alibaba_baselines = create_dummy_baselines()
print(f"\nLoaded results:")
print(f"GEFCom EEMD-TGNet models: {len(gefcom_models)}")
print(f"Alibaba EEMD-TGNet models: {len(alibaba_models)}")
print(f"GEFCom baseline models: {len(gefcom_baselines)}")
print(f"Alibaba baseline models: {len(alibaba_baselines)}")

Training results loaded successfully!
Available results:
  - gefcom_models
  - alibaba_models
  - gefcom_baselines
  - alibaba_baselines
  - device

Loaded results:
GEFCom EEMD-TGNet models: 3
Alibaba EEMD-TGNet models: 2
GEFCom baseline models: 3
Alibaba baseline models: 3
